# Build and Share a Fun ML App to Reveal Someone's Personality by How They Write


This notebook guides you through building a simple, interactive machine learning app using Gradio and Hugging Face Transformers.
The app will take a piece of text, analyze the writing style, and predict the likely MBTI personality type.


## Step 1, Load the Dataset

In [3]:

import pandas as pd

file_id = "1I4g7CvmYDHSn48SE7EXdLkiYqX-4gBse"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

df = pd.read_csv(url)
# df['posts'] = df['posts'].apply(lambda x: x.replace('|||', ' '))
# df['type'].unique()


In [ ]:
df.head()

## Suggested Visual 1: Show a few rows of the dataset

In [ ]:
df[['type', 'posts']].head()

## Step 2, Understanding the MBTI


The Myers-Briggs Type Indicator categorizes people into 16 types using four axes:
- Introversion or Extroversion
- Intuition or Sensing
- Thinking or Feeling
- Perceiving or Judging


## Step 3, Prepare the Data

In [ ]:

from sklearn.model_selection import train_test_split

df = df[df['posts'].notna()]
X_train, X_test, y_train, y_test = train_test_split(df['posts'], df['type'], test_size=0.2, random_state=42)


## Step 4, Train the Transformer Model

In [ ]:

!pip install transformers datasets --quiet

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

unique_labels = sorted(df['type'].unique())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

train_data = pd.DataFrame({'text': X_train, 'label': [label2id[y] for y in y_train]})
test_data = pd.DataFrame({'text': X_test, 'label': [label2id[y] for y in y_test]})

train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=16,
    id2label=id2label,
    label2id=label2id
)


In [ ]:

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    evaluation_strategy='epoch',
    logging_dir='./logs',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()


## Step 5, Build a Gradio Interface

In [ ]:

!pip install gradio --quiet

import gradio as gr
import torch

def predict_mbti(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    conf, pred_id = torch.max(probs, dim=1)
    pred_label = model.config.id2label[pred_id.item()]

    top5 = torch.topk(probs, k=5)
    result = "\n".join([f"{model.config.id2label[i]}: {probs[0][i]:.2f}" for i in top5.indices[0]])

    return f"**Predicted MBTI Type: {pred_label}**\n\nConfidence: {conf.item():.2f}\n\nTop predictions:\n{result}"

interface = gr.Interface(
    fn=predict_mbti,
    inputs=gr.Textbox(lines=6, placeholder="Write a paragraph about yourself"),
    outputs=gr.Markdown(),
    title="MBTI Personality Predictor",
    description="Enter your writing, and the model will guess your MBTI personality type."
)

interface.launch()
